# 03 Two-stage model (`two_stage_ffn_rf`): feed-forward network for the level, random forest for the flat: training and inference

The static + temporal network of `static_temporal_ffn.ipynb` multiplies the temporal and static inputs inside its hidden layers, so errors in how the market moves are carried into every flat. This notebook separates the two:
**the time effect and the flat effect are fitted by different models and combined without interaction.**

* **Stage 1 (level by month):** the temporal-features network of `temporal_ffn.ipynb` (`177 -> 64 -> 16 -> 1`, LeakyReLU), which predicts the rent level of each month from the temporal features (`temporal_scaled.csv`, scaled by the data pipeline).
  It is loaded from `results/models/temporal_ffn.pt`, not retrained here: run `temporal_ffn.ipynb` first.
* **Stage 2 (the flat):** a random forest on the static features (the 151 of the single-stage notebook) that predicts the **ratio** `rent / level of its month`. Because rents rose by about 50% over the period, a flat's premium over the average flat grows with the level,
  so the ratio is the right target (not the dollar difference). The fit is weighted by `level^2`, which makes the forest's squared error in ratio the squared error in dollars.
* **Prediction:** `level of the month x ratio of the flat`. The forest never sees the month, so the time effect cannot interact with the flat. The model is multiplicative, so **the growth of the same flats equals the growth of the stage-1 level by construction**:
  stage 2 cannot move the trend, and a bad trend passes straight through.

**One fit.** Both stages are fitted on the train months (2021-01..2024-03) only and never refitted on the validation months: the test rows have no rents, so the validation rows (2024-04..2025-03) are the only held-out rents the models can be compared on.
The same fitted model predicts the train rows (in-sample), the validation rows (held out), the 50,000 test rows (a forecast), and the rent of a fixed set of flats in every month (the flats of 2025-01..03, features only).

Model selection on the validation rows:

1. **Forest settings:** a small grid over the share of features per split and the minimum leaf size, scored with the true monthly level (a perfect stage 1) so that only stage 2 is judged.
2. **Stage-1 inputs:** the network on all 177 temporal features against networks on the 6 macro features and on macro + COE (18), over 5 seeds each, since a network's extrapolation depends on the seed. The choice is checked on the level of the
   validation months, on the two-stage error of the validation rows, and on the growth each network forecasts for 2026. The settings are picked on the validation rows, so the held-out predictions are slightly optimistic.

* **Outputs:** the forest in `results/models/` (the stage-1 network is `temporal_ffn.pt`), the model-selection tables in `results/tables/two_stage_ffn_rf/`, the predictions and the fixed-flat levels in `03_evaluation/predictions/`. Evaluation: `03_evaluation/evaluation.ipynb`.

In [1]:
import sys
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

import os, sys
from pathlib import Path
_here = Path(os.environ.get('RENT_PROJECT_ROOT') or Path.cwd()).resolve()
sys.path.insert(0, str(next(p for p in [_here, *_here.parents] if (p / 'data' / 'config.py').exists())))      # final_report: data/config.py holds every path
from data import config as cfg
sys.path.insert(0, str(cfg.MODELLING_DIR))     # 02_modelling: nn_utils.py is shared by the single-stage and two-stage notebooks
import nn_utils as nu

warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)
NAME = 'two_stage_ffn_rf'
SEEDS = range(5)
nu.use_results_dir(cfg.TWO_STAGE_NN_RESULTS_DIR)
nu.ensure_output_dirs(NAME)

## 1. Data

In [2]:
train, test, STATIC = nu.load_static()
fit, val = (train['sample_type'] == 'train').to_numpy(), (train['sample_type'] == 'val').to_numpy()
month, y = train['month'].to_numpy(), train[nu.LABEL].to_numpy()
X, X_test = train[STATIC].to_numpy(np.float32), test[STATIC].to_numpy(np.float32)
FIT_MONTHS, VALIDATION_MONTHS = np.arange(nu.TRAIN_LAST + 1), np.arange(nu.TRAIN_LAST + 1, nu.VALIDATION_LAST + 1)
month_mean = train.groupby('month')[nu.LABEL].mean()                                 # the actual level of each month 0..50
y_val, m_val = y[val], month[val]
fixed = np.isin(month, nu.Q1_2025)                                                   # the fixed flats: the rows of 2025-01..03 (their features only, never their rents)

temporal = nu.load_temporal()
ALL = list(temporal.columns)
MACRO = list(nu.MACRO_FEATURE)
MACRO_COE = MACRO + list(nu.COE_FEATURE)
ALL_NAME = f'network, all ({len(ALL)})'
STAGE1 = {'network, macro (6 features)': MACRO, 'network, macro + COE (18)': MACRO_COE, ALL_NAME: ALL}
print(f'static {len(STATIC)} features; fit rows {fit.sum():,} (months 0..{nu.TRAIN_LAST}), validation rows {val.sum():,} (months {nu.TRAIN_LAST + 1}..{nu.VALIDATION_LAST}), test rows {len(test):,}')
print({k: len(v) for k, v in STAGE1.items()})

static 151 features; fit rows 114,841 (months 0..38), validation rows 35,158 (months 39..50), test rows 50,000
{'network, macro (6 features)': 6, 'network, macro + COE (18)': 18, 'network, all (177)': 177}


## 2. Stage 1: the networks

The stage-1 model is the network of `temporal_ffn.ipynb` (all 177 features, seed 0), loaded from `results/models/temporal_ffn.pt`: it is not retrained here. For the comparison of the stage-1 inputs (section 4), the networks on the macro and macro + COE features and seeds 1..4 of the
177-feature network are fitted on the rows of months 0..38 with the same early stopping on the validation rows as `temporal_ffn.ipynb`. Every network is asked for the level of every month.

In [3]:
stage1_model, ckpt = nu.load_ffn(nu.MODEL_DIR / 'temporal_ffn.pt')                  # fitted by temporal_ffn.ipynb (run it first), not here
assert ckpt['temporal_features'] == ALL, 'temporal_ffn.pt was fitted on other temporal features'
FIT_ARGS = dict(epochs=stage1_model['config']['epochs'], patience=stage1_model['config']['patience'])      # the comparison networks stop early by the same rule
held = {name: {} for name in STAGE1}
for name, features in STAGE1.items():
    Z = temporal[features]
    for seed in SEEDS:
        if name == ALL_NAME and seed == 0:
            held[name][seed] = nu.month_levels(stage1_model, Z)
        else:
            held[name][seed] = nu.month_levels(nu.fit_ffn(Z, month[fit], y[fit], valid=(m_val, None, y_val), seed=seed, **FIT_ARGS), Z)
level = held[ALL_NAME][0]                                                             # stage-1 level of the 67 months
level_w = level.loc[month[fit]].to_numpy()

## 3. Stage 2: forest settings

The forest is fitted on the rows of months 0..38: target = rent / level of its month (the in-sample level of the seed-0 network above), weighted by `level^2`. (The in-sample level of every stage-1 choice is within a few dollars of the month's
actual mean rent, so the same forest serves all of them.) Grid: the share of features tried per split and the minimum leaf size (200 trees, depth 24, 60% of the rows per tree), scored on the validation rows with the **oracle level**, the true monthly mean of each
validation month, i.e. a perfect stage 1.

In [4]:
oracle = month_mean.loc[m_val].to_numpy()
BASE_PARAMS = {'n_estimators': 200, 'max_depth': 24, 'max_samples': 0.6}
grid = []
for max_features in (0.33, 0.55, 0.85):
    for min_samples_leaf in (5, 10, 20):
        forest = RandomForestRegressor(n_jobs=-1, random_state=0, max_features=max_features, min_samples_leaf=min_samples_leaf, **BASE_PARAMS
                                       ).fit(X[fit], y[fit] / level_w, sample_weight=level_w ** 2)
        grid.append({'max_features': max_features, 'min_samples_leaf': min_samples_leaf, 'validation_rmse_oracle_level': nu.rmse(y_val, oracle * forest.predict(X[val]))})
grid = pd.DataFrame(grid).sort_values('validation_rmse_oracle_level').reset_index(drop=True)
print(grid.round(2).to_string())
RF_PARAMS = {**BASE_PARAMS, 'max_features': float(grid.loc[0, 'max_features']), 'min_samples_leaf': int(grid.loc[0, 'min_samples_leaf'])}
print('\nforest parameters:', RF_PARAMS)

   max_features  min_samples_leaf  validation_rmse_oracle_level
0          0.33                10                        469.07
1          0.55                10                        469.12
2          0.85                20                        469.50
3          0.55                20                        469.54
4          0.85                10                        469.60
5          0.33                 5                        469.85
6          0.33                20                        470.02
7          0.55                 5                        470.34
8          0.85                 5                        470.73

forest parameters: {'n_estimators': 200, 'max_depth': 24, 'max_samples': 0.6, 'max_features': 0.33, 'min_samples_leaf': 10}


## 4. The two-stage model, and the choice of the stage-1 inputs

The forest with the chosen settings, fitted on the train rows, with the stage-1 level of the seed-0 network on all 177 features: `level of the month x ratio of the flat`. It predicts the train, validation and test rows; the level of the same flats in each month is
level(month) x the mean ratio of the fixed flats.

The same forest is combined with every stage-1 choice to compare them (`stage1_selection.csv`): the error of the level of months 39..50, the two-stage RMSE on the validation rows, and the growth at mid-2026 (2026-06 over 2025-01..03) each network forecasts (5 seeds).
"carry-forward" repeats the actual level of month 38; "straight-line" is a linear trend of the monthly mean rent of months 0..38.

In [5]:
forest = RandomForestRegressor(n_jobs=-1, random_state=0, **RF_PARAMS).fit(X[fit], y[fit] / level_w, sample_weight=level_w ** 2)
ratio_val = forest.predict(X[val])
pred_train, pred_val = level.loc[month[fit]].to_numpy() * forest.predict(X[fit]), level.loc[m_val].to_numpy() * ratio_val
pred_test = level.loc[test['month']].to_numpy() * forest.predict(X_test)
level_fixed = level * forest.predict(X[fixed]).mean()                                  # the same flats in every month: level(m) x mean ratio of the flats
print(f'in-sample ratio mean {forest.predict(X[fit]).mean():.3f}; validation ratio mean {ratio_val.mean():.3f}')

two_stage_rmse = lambda levels: nu.rmse(y_val, levels.loc[m_val].to_numpy() * ratio_val)
growth = lambda lv: lv.loc[65] / lv.loc[nu.Q1_2025].mean() - 1                       # 2026-06 over 2025-01..03
def level_scores(levels):
    err = levels.loc[VALIDATION_MONTHS] - month_mean.loc[VALIDATION_MONTHS]
    return {'level_rmse': float(np.sqrt((err ** 2).mean())), 'level_bias': float(err.mean()), 'last_month_error': float(err.iloc[-1])}

slope, intercept = np.polyfit(FIT_MONTHS, month_mean.loc[FIT_MONTHS], 1)
carry, trend = pd.Series(month_mean.loc[nu.TRAIN_LAST], index=range(nu.N_MONTHS)), pd.Series(intercept + slope * np.arange(nu.N_MONTHS))
selection = {'oracle level (true monthly mean)': {'two_stage_rmse': two_stage_rmse(month_mean)},
             'carry-forward (month 38)': {**level_scores(carry), 'two_stage_rmse': two_stage_rmse(carry)},
             'straight-line trend': {**level_scores(trend), 'two_stage_rmse': two_stage_rmse(trend)}}
for name in STAGE1:
    scores, rmses, g = pd.DataFrame([level_scores(held[name][s]) for s in SEEDS]), np.array([two_stage_rmse(held[name][s]) for s in SEEDS]), np.array([growth(held[name][s]) for s in SEEDS])
    selection[name] = {**scores.mean().to_dict(), 'level_rmse_sd': scores['level_rmse'].std(), 'two_stage_rmse': rmses.mean(), 'two_stage_rmse_sd': rmses.std(ddof=1),
                       'growth_mid_2026_mean': g.mean(), 'growth_mid_2026_min': g.min(), 'growth_mid_2026_max': g.max()}
selection = pd.DataFrame(selection).T
print(selection.round(3).to_string())
print(level_fixed.loc[44:].round(0).rename(lambda m: nu.month_label(m)[0]).to_frame('level of the same flats').T.to_string())

in-sample ratio mean 0.995; validation ratio mean 0.996
                                  two_stage_rmse  level_rmse  level_bias  last_month_error  level_rmse_sd  two_stage_rmse_sd  growth_mid_2026_mean  growth_mid_2026_min  growth_mid_2026_max
oracle level (true monthly mean)         469.074         NaN         NaN               NaN            NaN                NaN                   NaN                  NaN                  NaN
carry-forward (month 38)                 471.358      38.403     -28.990           -65.902            NaN                NaN                   NaN                  NaN                  NaN
straight-line trend                      598.768     383.662     368.448           523.506            NaN                NaN                   NaN                  NaN                  NaN
network, macro (6 features)              543.685     256.941    -214.505          -440.687         75.823             40.541                 0.037               -0.046                0.121

## 5. Save the models, the model-selection tables and the predictions

In [6]:
paths = [nu.MODEL_DIR / f'{NAME}_forest.joblib']
joblib.dump({'forest': forest, 'static_features': STATIC, 'rf_params': RF_PARAMS, 'target': 'rent / stage-1 level of the month, weighted by level^2', 'stage1_model': 'temporal_ffn.pt'}, paths[0], compress=3)
paths += [nu.save_table(grid, NAME, 'rf_tuning.csv', index=False), nu.save_table(selection, NAME, 'stage1_selection.csv', index_label='stage 1')]
paths += [nu.save_predictions(NAME, 'train', train[fit], pred_train),      # in-sample: the 114,841 train rows
          nu.save_predictions(NAME, 'val', train[val], pred_val),          # held out: the 35,158 validation rows
          nu.save_predictions(NAME, 'test', test, pred_test),              # the 50,000 test rows
          nu.save_levels(NAME, level_fixed)]
print('saved:', *paths, sep='\n  ')

saved:
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/02_modelling/two_stage_models/neural_network/results/models/two_stage_ffn_rf_forest.joblib
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/02_modelling/two_stage_models/neural_network/results/tables/two_stage_ffn_rf/rf_tuning.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/02_modelling/two_stage_models/neural_network/results/tables/two_stage_ffn_rf/stage1_selection.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/03_evaluation/predictions/train/two_stage_ffn_rf.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/03_evaluation/predictions/val/two_stage_ffn_rf.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/03_evaluation/predictions/test/two_stage_ffn_rf.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/03_evaluation/predictions/levels/two_stage_ffn_rf.csv
